# TF-IDF Workshop: From Raw Text to Classification
A guided, section-by-section practical using the UCI SMS Spam Collection.

**Task:** classify SMS messages as `ham` (legitimate) or `spam`.

Run the notebook from top to bottom. Imports are introduced in the section where they are first needed, rather than collected at the top.

## Learning objectives
- Inspect and clean a labelled text dataset.
- Calculate TF, document frequency and smoothed IDF manually.
- Convert text into TF-IDF vectors.
- Train and evaluate a Logistic Regression classifier.
- Predict new messages and save the fitted pipeline components.

In [ ]:
from pathlib import Path

# Keep the extracted SMSSpamCollection file beside this notebook, or in ./data/.
PROJECT_DIR = Path.cwd()
DATA_CANDIDATES = [PROJECT_DIR / "SMSSpamCollection", PROJECT_DIR / "data" / "SMSSpamCollection"]
DATA_FILE = next((path for path in DATA_CANDIDATES if path.is_file()), None)
MODEL_FILE = PROJECT_DIR / "tfidf_spam_classifier.joblib"

if DATA_FILE is None:
    searched = "\n".join(str(path) for path in DATA_CANDIDATES)
    raise FileNotFoundError("SMSSpamCollection was not found. Place the extracted file beside this notebook or inside a data folder. Searched:\n" + searched)

print(f"Using local dataset: {DATA_FILE.resolve()}")
print(f"Model will be saved to: {MODEL_FILE.resolve()}")


## 1. Load and inspect the local dataset
This notebook uses your already-downloaded, extracted `SMSSpamCollection` file. It does **not** download anything from the internet.

Place the file in the same folder as this notebook or in a subfolder named `data`.


In [ ]:
def load_dataset():
    import pandas as pd
    df = pd.read_csv(DATA_FILE, sep="\t", header=None, names=["label", "text"], encoding="utf-8", on_bad_lines="skip")
    df = df.dropna(subset=["label", "text"])
    df["label"] = df["label"].astype(str).str.strip().str.lower()
    df["text"] = df["text"].astype(str).str.strip()
    df = df[df["label"].isin(["ham", "spam"])]
    df = df[df["text"] != ""]
    return df.drop_duplicates(subset=["label", "text"]).reset_index(drop=True)

def explore_dataset(df):
    print(f"Number of messages: {len(df):,}")
    display(df.head())
    display(df["label"].value_counts().rename("count").to_frame())
    print("Class percentages:")
    display((df["label"].value_counts(normalize=True) * 100).round(2).rename("percentage").to_frame())

print(f"Loading dataset from: {DATA_FILE.resolve()}")


In [ ]:
df = load_dataset()
explore_dataset(df)

## 2. Calculate TF-IDF manually
Use three tiny documents to see the vocabulary, document frequency, IDF and TF-IDF weights.

In [ ]:
def manual_tfidf_demo():
    import math

    documents = ["payment delayed", "payment not processed", "forex fee charged"]
    tokenized_documents = [document.lower().split() for document in documents]
    vocabulary = sorted(set(word for document in tokenized_documents for word in document))
    number_of_documents = len(tokenized_documents)

    print("Vocabulary:", vocabulary)
    print("TF(t,d) = count(t in d) / total terms in d")
    print("IDF(t) = log((1 + N) / (1 + df(t))) + 1")
    print("TF-IDF = TF x IDF\n")

    document_frequency = {
        term: sum(term in document for document in tokenized_documents)
        for term in vocabulary
    }
    idf_values = {
        term: math.log((1 + number_of_documents) / (1 + document_frequency[term])) + 1
        for term in vocabulary
    }

    print("Term | Document frequency | IDF")
    for term in vocabulary:
        print(f"{term:<12} {document_frequency[term]:>3} {idf_values[term]:>10.4f}")

    print("\nTF-IDF values before L2 normalisation:")
    for index, document in enumerate(tokenized_documents, start=1):
        scores = {}
        for term in vocabulary:
            tf = document.count(term) / len(document)
            score = tf * idf_values[term]
            if score > 0:
                scores[term] = round(score, 4)
        print(f"D{index}: {scores}")

manual_tfidf_demo()

## 3. Split into training and test sets
We split before learning the vocabulary and IDF weights. `stratify=y` preserves approximately the same class proportions in both sets.

In [ ]:
def split_dataset(df):
    from sklearn.model_selection import train_test_split
    X = df["text"]
    y = df["label"]
    return train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

X_train, X_test, y_train, y_test = split_dataset(df)
print(f"Training messages: {len(X_train):,}")
print(f"Testing messages : {len(X_test):,}")

## 4. Convert text into TF-IDF vectors
Fit the vectorizer on training text only. Transform the test text with the fitted vocabulary and IDF weights to avoid information leakage.

In [ ]:
def create_tfidf_features(X_train, X_test):
    from sklearn.feature_extraction.text import TfidfVectorizer
    vectorizer = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
    )
    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_test_tfidf = vectorizer.transform(X_test)
    return vectorizer, X_train_tfidf, X_test_tfidf

vectorizer, X_train_tfidf, X_test_tfidf = create_tfidf_features(X_train, X_test)
print("Training matrix:", X_train_tfidf.shape)
print("Testing matrix :", X_test_tfidf.shape)
print("Vocabulary size:", len(vectorizer.get_feature_names_out()))
print("First 25 terms:", vectorizer.get_feature_names_out()[:25].tolist())

sample_vector = X_train_tfidf[0].toarray().ravel()
top_indices = sample_vector.argsort()[-8:][::-1]
print("\nSample message:", X_train.iloc[0])
print("Top weighted terms:")
for index in top_indices:
    if sample_vector[index] > 0:
        print(f"{vectorizer.get_feature_names_out()[index]}: {sample_vector[index]:.4f}")

## 5. Establish a simple baseline
A majority-class baseline always predicts the most common label. A useful model should do better than this simple reference.

In [ ]:
def run_baseline(X_train_tfidf, y_train, X_test_tfidf, y_test):
    from sklearn.dummy import DummyClassifier
    from sklearn.metrics import accuracy_score
    baseline = DummyClassifier(strategy="most_frequent")
    baseline.fit(X_train_tfidf, y_train)
    predictions = baseline.predict(X_test_tfidf)
    print(f"Majority-class baseline accuracy: {accuracy_score(y_test, predictions):.4f}")
    return baseline

baseline = run_baseline(X_train_tfidf, y_train, X_test_tfidf, y_test)

## 6. Train Logistic Regression
Logistic Regression is a strong, interpretable baseline for sparse TF-IDF text features.

In [ ]:
def train_classifier(X_train_tfidf, y_train):
    from sklearn.linear_model import LogisticRegression
    model = LogisticRegression(max_iter=1000, class_weight="balanced")
    model.fit(X_train_tfidf, y_train)
    return model

model = train_classifier(X_train_tfidf, y_train)
print("Model training complete.")

## 7. Evaluate on unseen test data
Inspect precision, recall, F1-score and the confusion matrix. Accuracy alone can hide poor performance on the minority spam class.

In [ ]:
def evaluate_model(model, X_test_tfidf, y_test):
    from sklearn.metrics import (
        accuracy_score, precision_score, recall_score, f1_score,
        classification_report, confusion_matrix
    )
    predictions = model.predict(X_test_tfidf)
    print(f"Accuracy      : {accuracy_score(y_test, predictions):.4f}")
    print(f"Spam precision: {precision_score(y_test, predictions, pos_label='spam', zero_division=0):.4f}")
    print(f"Spam recall   : {recall_score(y_test, predictions, pos_label='spam', zero_division=0):.4f}")
    print(f"Spam F1-score : {f1_score(y_test, predictions, pos_label='spam', zero_division=0):.4f}")
    print("\nClassification report:")
    print(classification_report(y_test, predictions, digits=4, zero_division=0))
    print("Confusion matrix (rows/columns ordered ham, spam):")
    print(confusion_matrix(y_test, predictions, labels=["ham", "spam"]))
    return predictions

test_predictions = evaluate_model(model, X_test_tfidf, y_test)

## 8. Predict new messages
The messages below were not part of the training set. Their text is transformed using the already-fitted vectorizer.

In [ ]:
new_messages = [
    "Congratulations! You have won a free prize. Call now to claim.",
    "Hey, are we still meeting for lunch today?",
    "URGENT! Your account has been selected for a cash reward.",
    "Please send me the project report when you get time.",
]

new_vectors = vectorizer.transform(new_messages)
new_predictions = model.predict(new_vectors)
new_probabilities = model.predict_proba(new_vectors)
classes = list(model.classes_)

for message, prediction, probability_row in zip(new_messages, new_predictions, new_probabilities):
    spam_probability = probability_row[classes.index("spam")]
    print(f"\nMessage: {message}")
    print(f"Prediction: {prediction.upper()}")
    print(f"Estimated spam probability: {spam_probability:.1%}")

## 9. Save the fitted vectorizer and model
Both objects are needed for future predictions: the vectorizer converts incoming text into the same feature space, and the classifier predicts the label.

In [ ]:
def save_model(vectorizer, model):
    import joblib
    joblib.dump({"vectorizer": vectorizer, "model": model}, MODEL_FILE)
    print(f"Saved to: {MODEL_FILE}")

save_model(vectorizer, model)

## Experiments for the team
1. Change `ngram_range=(1, 2)` to `(1, 1)`. Compare vocabulary size and metrics.
2. Try `(1, 3)`. Does the extra vocabulary improve results?
3. Change `min_df` from 2 to 3 or 5. What happens to vocabulary size?
4. Compare Logistic Regression with Multinomial Naive Bayes.
5. Discuss why precision and recall matter for spam detection.

**Important:** keep the same train/test split when comparing experiments.

## Dataset reference
Almeida, T. & Hidalgo, J. (2011). *SMS Spam Collection*. UCI Machine Learning Repository. https://doi.org/10.24432/C5CC84. License: CC BY 4.0.